In [1]:
import requests
import pandas as pd
import pyarrow 

url = "https://api.open-meteo.com/v1/forecast"


load_date = "2026-01-01"

response = requests.get(url,
    params={
        "lstitude": 55.7522,
        "longitude": 37.6156,
        "hourly": ["temperature_2m", "rain"],
        "start_date": load_date,
        "end_date": load_date, 
    },
)

# response.raise_for_status()
print(response.status_code)

data = response.json()

df = pd.json_normalize(data)

csv_path = f"moscow_{load_date}.csv"
df.to_csv(csv_path, index=False)

parquet_path = f"moscow_{load_date}.parquet"
df.to_parquet(parquet_path, index=False, engine="pyarrow")

df.head()

400


,reason,error
0,Parameter 'latitude' and 'longitude' must have...,True


In [14]:
import requests
import pandas as pd
import pyarrow 
from datetime import datetime, timedelta
import os

url = "https://api.open-meteo.com/v1/forecast"

update_at = datetime.now().date()

params = {
    "latitude": 59.9386,
    "longitude": 30.3141,
    "daily": [
        "temperature_2m_mean",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "weather_code",
    ],
    "timezone": "Europe/Moscow",
    "start_date": '2026-01-12',
    "end_date": '2026-01-19',
}

response = requests.get(url, params=params)
print(response.status_code)

data = response.json()

daily_data = data["daily"]

df = pd.DataFrame(daily_data)

df["business_date"] = pd.to_datetime(df["time"])
df.drop(columns=["time"], inplace=True)

df["update_at"] = update_at

csv_path = f"tmp/piter_daily_last_week_{update_at}.csv"
df.to_csv(csv_path, index=False)

parquet_path = f"tmp/piter_daily_last_week_{update_at}.parquet"
df.to_parquet(parquet_path, index=False, engine="pyarrow")

df.head()

200


,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,weather_code,business_date,update_at
0,-8.2,0.0,0.0,0.0,3,2026-01-12,2026-01-18
1,-7.6,0.0,0.0,0.0,3,2026-01-13,2026-01-18
2,-7.1,0.0,0.0,0.0,3,2026-01-14,2026-01-18
3,-6.4,0.0,0.0,0.0,3,2026-01-15,2026-01-18
4,-7.7,0.0,0.0,0.0,3,2026-01-16,2026-01-18


In [ ]:
# 1. Обязательно добавить поля update_at(дата выгрузки данных) и также бизнес дату(под бизнес датой понимается дата, когда произошло какое-то событие, обычно уже присутствует в данных), если есть
# 2. Данные из API должны сохраняться в локальную папку tmp/ вашего рабочего пространства VSCode в формате csv
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry
from datetime import datetime
import os

os.makedirs("tmp", exist_ok=True)

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

load_date = datetime.now().date()

url = "https://api.open-meteo.com/v1/forecast"

daily_vars = [
    "temperature_2m_mean",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "weather_code",
]
params = {
	"latitude": 59.9386,
	"longitude": 30.3141,
	"daily": daily_vars,
    "timezone": "Europe/Moscow",
    "start_date": '2026-01-12',
    "end_date": '2026-01-19',
}
responses = openmeteo.weather_api(url, params=params)
response = responses[0]

data_daily = response.Daily()

n_days = len(data_daily.Variables(0).ValuesAsNumpy())
stuct_data_daily = {"date": pd.date_range(
    start=pd.to_datetime(data_daily.Time(), unit="s", utc = True),
    periods=n_days
)}

for i, var_name in enumerate(daily_vars):
    stuct_data_daily[var_name] = data_daily.Variables(i).ValuesAsNumpy()

df_daily = pd.DataFrame(stuct_data_daily)

df_daily["business_date"] = df_daily["date"].dt.date
df_daily["update_at"] = load_date
df_daily.head(20)

csv_path = f"tmp/piter_daily_last_week_{load_date.strftime('%Y-%m-%d')}.csv"
df_daily.to_csv(csv_path, index=False)